# 03 · Prototype run — 12 monthly LULC maps for any location

**Input:** latitude, longitude, start month  
**Processing:** 12 Sentinel-1 VV/VH monthly composites from Earth Engine → trained U-Net  
**Output:** class share per month (%), month-to-month change and first-to-last change (percentage points), maps

The same pipeline backs the Streamlit app (`streamlit run app.py`) and the CLI (`python run_lulc.py`).

In [ ]:
import os, sys
from pathlib import Path

import pandas as pd
from IPython.display import Image, display

PROJECT_DIR = next(p for p in [Path.cwd(), *Path.cwd().parents] if (p / "src").is_dir())
sys.path.insert(0, str(PROJECT_DIR))
from src import pipeline

pd.set_option("display.precision", 2)
EE_PROJECT = os.environ.get("EE_PROJECT")   # or "your-gcp-project-id"

## Inputs

In [ ]:
LATITUDE    = 48.7758      # Stuttgart city centre
LONGITUDE   = 9.1829
START_MONTH = "2025-10"    # analysis covers this month + the next 11
AREA_KM     = 2.0

## Run

In [ ]:
run = pipeline.run(
    LATITUDE, LONGITUDE, START_MONTH, area_km=AREA_KM, ee_project=EE_PROJECT,
    progress=lambda msg, frac: print(f"[{(frac or 0) * 100:5.1f}%] {msg}"),
)
res, figs = run["results"], run["figures"]
print("\nAcquisition:", run["metadata"]["acquisition"]["note"])
print("Scenes per month:", run["metadata"]["acquisition"]["scene_counts"])
print("Outputs:", run["run_dir"])

## LULC map of the location over 12 months

In [ ]:
display(Image(filename=str(figs["lulc_12_months.png"])))

In [ ]:
display(Image(filename=str(figs["lulc_12_months.gif"])))

## Share of the area covered by each class (%)

In [ ]:
display(res["percentages"].round(2))
display(Image(filename=str(figs["class_composition.png"])))

## Month-to-month change (percentage points)

Each class column is the change in that class's share relative to the previous month.
`Pixels changed (%)` is the gross share of pixels that switched class.

In [ ]:
display(res["month_to_month_pp"].round(2))
display(Image(filename=str(figs["month_to_month_change_pp.png"])))

## Total change, first month → last month (percentage points)

In [ ]:
display(res["total_change_pp"].round(2))
display(Image(filename=str(figs["total_change_pp.png"])))

## Spatial change

In [ ]:
display(Image(filename=str(figs["change_maps.png"])))
s = res["summary"]
print(f"Pixels with a different class in the last vs first month: {s['changed_pct_first_last']:.2f} %"
      f"  ({s['changed_area_ha_first_last']:.1f} ha)")
print(f"Pixels stable in all 12 months: {s['stable_all_months_pct']:.2f} %")
display(res["transition_first_last"])

## A second urban area

Same call, different city: Frankfurt am Main (Ostend / EZB area).

In [ ]:
run2 = pipeline.run(50.1120, 8.7050, START_MONTH, area_km=AREA_KM, ee_project=EE_PROJECT)
display(Image(filename=str(run2["figures"]["lulc_12_months.png"])))
display(run2["results"]["total_change_pp"].round(2))

### Reading the results

The model sees radar backscatter only. Month-to-month differences mix real change
(construction, harvest, flooding) with seasonal effects on the signal (soil moisture,
vegetation water content, snow). Small swings of 1–3 pp between months in stable
urban areas are mostly the latter.